# nuPlan Dataset

In [38]:
import os
NUPLAN_DATA_ROOT = os.getenv('NUPLAN_DATA_ROOT', '/mnt/zen-storage/ml/open-data/nuplan/dataset')

In [39]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from nuplan.database.nuplan_db.db_cli_queries import (
    _get_table_row_count_from_db, 
    get_unique_frame_count,
    get_unique_frames_by_scenario_type
)

from tqdm import tqdm

def process_db_file(db_file):
    try:
        scenario_frame_count = get_unique_frames_by_scenario_type(db_file)
        tagged_frame_count = get_unique_frame_count(db_file)
        frame_count = _get_table_row_count_from_db(db_file, 'lidar_pc')
        return scenario_frame_count, tagged_frame_count, frame_count
    except Exception as e:
        return db_file, e

def get_scenario_count(db_files, workers=20):
    scenario_frames = defaultdict(int)
    total_tagged_frame_count = 0
    total_frame_count = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(process_db_file, db_file) for db_file in db_files]
        for future in tqdm(futures, total=len(futures), desc="Processing database files"):
            scenario_frame_count, tagged_frame_count, frame_count = future.result()

            for scenario_type, count in scenario_frame_count:
                scenario_frames[scenario_type] += count

            total_tagged_frame_count += tagged_frame_count
            total_frame_count += frame_count

    return scenario_frames, total_tagged_frame_count, total_frame_count

# Mini Dataset

In [40]:
split = "mini"
nuplan_split_dir = f"{NUPLAN_DATA_ROOT}/nuplan-v1.1/splits/{split}"

db_files = []
for file in os.listdir(nuplan_split_dir):
    if file.endswith('.db'):
        db_files.append(os.path.join(nuplan_split_dir, file))

scenario_frames, total_tagged_frame_count, total_frame_count = get_scenario_count(db_files)

Processing database files:   0%|          | 0/64 [00:00<?, ?it/s]

Processing database files: 100%|██████████| 64/64 [00:00<00:00, 198.64it/s]


In [41]:
import pandas as pd


df = pd.DataFrame({
    'scenario_tag': list(scenario_frames.keys()), 
    'scenario_count': list(scenario_frames.values())
})

df.sort_values(by='scenario_count', ascending=False, inplace=True)

## Frames and Scenario Tags

Below is the total frame count, and the total tagged frames (frames having a scenario tag). A frame can have multiple scenario tags A frame can have multiple scenario tags.

### Frame Count

In [42]:
print(f"{total_frame_count:,}")

518,999


### Tagged Frames

In [43]:
scenario_tag_count = df['scenario_count'].sum()
print(f"{total_tagged_frame_count:,}")

390,186


### Scenario Tags

Here we see that the sum of unique frames for each tag is greater than the total frame count, so the relationship between frames and scenario-tags must be one-to-many.

In [44]:
total_scenario_tags = df['scenario_count'].sum()
print(f"{total_scenario_tags:,}")

821,831


Scenario Distribution

In [45]:
df['percentage_of_total_frames'] = ((df['scenario_count'] / total_frame_count) * 100).round(2)
df['percentage_of_scenario_frames'] = ((df['scenario_count'] / total_tagged_frame_count) * 100).round(2)
df

,scenario_tag,scenario_count,percentage_of_total_frames,percentage_of_scenario_frames
1,stationary,188367,36.29,48.28
5,on_intersection,84376,16.26,21.62
0,on_pickup_dropoff,78646,15.15,20.16
4,traversing_intersection,57786,11.13,14.81
7,on_traffic_light_intersection,57415,11.06,14.71
...,...,...,...,...
39,changing_lane_to_left,15,0.00,0.00
58,changing_lane_to_right,7,0.00,0.00
62,high_magnitude_jerk,7,0.00,0.00
51,behind_bike,2,0.00,0.00


In [46]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Table(
        header=dict(
            values=[
                "Scenario Tag", 
                "Scenario Count", 
                "Percentage of Total Frames", 
                "Percentage of Frames with a Scenario Tag"
            ]
        ),
        cells=dict(
            values=[
                df['scenario_tag'], 
                df['scenario_count'], 
                df['percentage_of_total_frames'],
                df['percentage_of_scenario_frames']
            ]
        )
    )
)

fig.update_layout(
    title='Scenario Tags',
    height=800,
    showlegend=False,
)
fig.show()

In [47]:
df['cumulative_percentage_of_total_scenario_tags'] = 100 * df['scenario_count'].cumsum() / df['scenario_count'].sum()

df_top = df.head(30)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_top['scenario_tag'],
    y=df_top['scenario_count'],
    name='Scenario Tag Count'
))
fig.add_trace(go.Scatter(
    x=df_top['scenario_tag'],
    y=df_top['cumulative_percentage_of_total_scenario_tags'],
    name='Cumulative %',
    yaxis='y2',
    line=dict(color='red', width=2)
))
fig.update_layout(
    title='Pareto Chart of Scenario Tags (Top 30)',
    yaxis=dict(title='Count'),
    yaxis2=dict(
        title='Cumulative %',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    legend=dict(x=0.7, y=1.1),
)
fig.show()

In [48]:
df_middle = df.iloc[20:40]
fig = go.Figure()
fig.add_trace(go.Bar(x=df_middle['scenario_tag'], y=df_middle['scenario_count'], name='Middle 20 Scenario Count'))
fig.update_layout(
    title='Frame Count for Middle 20 Scenarios', 
    xaxis_title='Scenario Tag', 
    yaxis_title='Percentage of Total Frames'
)
fig.show()

In [49]:
df_bottom = df.iloc[40:]
fig = go.Figure()
fig.add_trace(go.Bar(x=df_bottom['scenario_tag'], y=df_bottom['scenario_count'], name='Bottom 20 Scenario Count'))
fig.update_layout(
    title='Frame Count for Bottom 20 Scenarios', 
    xaxis_title='Scenario Tag', 
    yaxis_title='Percentage of Total Frames'
)
fig.show()

In [50]:
from nuplan.database.nuplan_db_orm.nuplandb import NuPlanDB
from nuplan.database.nuplan_db.query_session import execute_many, execute_one
from collections import defaultdict


def analyze_all_scenario_tags(db_file: str):
    """
    Analyze consecutive frames for each scenario tag type in the database.

    Args:
        db_file: The path to the database file to analyze.
    
    Returns:
        A dictionary mapping scenario tag types to their consecutive frame statistics.
    """
    # Connect to the database
    db = NuPlanDB(data_root=NUPLAN_DATA_ROOT, load_path=db_file)

    # Get all unique scenario tag types
    scenario_tags = db.get_unique_scenario_tags()

    results = defaultdict(dict)

    for tag_type in scenario_tags:
        frames = get_frames_with_scenario_tag(db_file, tag_type)

        if not frames:
            results[tag_type] = {
                "total_frames": 0,
                "sequences": [],
                "max_consecutive": 0,
                "avg_consecutive": 0,
            }
            continue

        # Count consecutive frames for each sequence
        consecutive_counts = count_consecutive_frames(frames)

        # Store statistics
        results[tag_type] = {
            "total_frames": len(frames),
            "sequences": consecutive_counts,
            "max_consecutive": max(consecutive_counts) if consecutive_counts else 0,
            "avg_consecutive": sum(consecutive_counts) / len(consecutive_counts) \
                if consecutive_counts else 0,
        }

    return results


def get_frames_with_scenario_tag(db_file, scenario_tag_type):
    """
    Get frames with a specific scenario tag along with their scene context/
    """
    query = """
    SELECT
        lp.token,
        lp.timestamp,
        lp.scene_token,
        lp.next_token,
        s.name as scene_name
    FROM lidar_pc as lp
    INNER JOIN scenario_tag as st ON lp.token = st.lidar_pc_token
    INNER JOIN scene AS s ON lp.scene_token = s.token
    WHERE st.type = ?
    ORDER BY s.name, lp.timestamp ASC;
    """

    result = []
    for row in execute_many(query, (scenario_tag_type,), db_file):
        result.append({
            "token": row["token"],
            "timestamp": row["timestamp"],
            "scene_token": row["scene_token"],
            "next_token": row["next_token"],
            "scene_name": row["scene_name"],
        })

    return result

def count_consecutive_frames(frames_list):
    """
    Count consecutive frames in a list of frames.
    """
    if not frames_list:
        return []
    
    consecutive_counts = []
    current_count = 1

    for i in range(1, len(frames_list)):
        prev_next_token = frames_list[i - 1]["next_token"]
        curr_token = frames_list[i]["token"]
        prev_scene = frames_list[i - 1]["scene_token"]
        curr_scene = frames_list[i]["scene_token"]

        if curr_scene == prev_scene and curr_token == prev_next_token:
            current_count += 1
        else:
            consecutive_counts.append(current_count)
            current_count = 1
    
    consecutive_counts.append(current_count)

    return consecutive_counts

In [51]:
db_file = db_files[0]

db = NuPlanDB(data_root=NUPLAN_DATA_ROOT, load_path=db_file)

scenario_tags = set()

for db_file in db_files:
    db = NuPlanDB(data_root=NUPLAN_DATA_ROOT, load_path=db_file)
    scenario_tags.update(db.get_unique_scenario_tags())

print(scenario_tags)

{'traversing_intersection', 'low_magnitude_speed', 'near_pedestrian_on_crosswalk', 'on_stopline_traffic_light', 'near_barrier_on_driveable', 'medium_magnitude_speed', 'starting_right_turn', 'behind_long_vehicle', 'behind_pedestrian_on_pickup_dropoff', 'behind_pedestrian_on_driveable', 'on_carpark', 'high_magnitude_speed', 'accelerating_at_stop_sign_no_crosswalk', 'traversing_crosswalk', 'accelerating_at_stop_sign', 'on_pickup_dropoff', 'stationary_at_traffic_light_with_lead', 'high_magnitude_jerk', 'near_long_vehicle', 'on_all_way_stop_intersection', 'near_multiple_vehicles', 'changing_lane_to_right', 'accelerating_at_traffic_light_with_lead', 'starting_protected_noncross_turn', 'starting_low_speed_turn', 'starting_straight_stop_sign_intersection_traversal', 'starting_left_turn', 'near_high_speed_vehicle', 'near_trafficcone_on_driveable', 'stationary_at_crosswalk', 'traversing_traffic_light_intersection', 'stationary', 'starting_unprotected_cross_turn', 'starting_unprotected_noncross_t

In [52]:
turn_tags = ["starting_right_turn", "starting_left_turn"]

total_frames = 0
consecutive_counts = []
for db_file in db_files:
    db = NuPlanDB(data_root=NUPLAN_DATA_ROOT, load_path=db_file)
    for tag in turn_tags:
        frames = get_frames_with_scenario_tag(db_file, tag)
        consecutive_counts.extend(count_consecutive_frames(frames))
        total_frames += len(frames)

print(f"Total Frames: {total_frames}")
print(f"Max Consecutive: {max(consecutive_counts) if consecutive_counts else 0}")
print(f"Average Consecutive: {sum(consecutive_counts) / len(consecutive_counts) if consecutive_counts else 0}")

Total Frames: 1204
Max Consecutive: 20
Average Consecutive: 11.466666666666667


In [53]:
total_frames = defaultdict(int)
consecutive_counts = defaultdict(list)

for db_file in db_files:
    db = NuPlanDB(data_root=NUPLAN_DATA_ROOT, load_path=db_file)
    for tag in scenario_tags:
        frames = get_frames_with_scenario_tag(db_file, tag)
        consecutive_counts[tag].extend(count_consecutive_frames(frames))
        total_frames[tag] += len(frames)

scenario_tag_stats = defaultdict(list)
for tag, counts in consecutive_counts.items():
    scenario_tag_stats["tag"].append(tag)
    scenario_tag_stats["total_frames"].append(total_frames[tag])
    scenario_tag_stats["max_consecutive"].append(max(counts) if counts else 0)
    scenario_tag_stats["min_consecutive"].append(min(counts) if counts else 0)
    scenario_tag_stats["avg_consecutive"].append(sum(counts) / len(counts) if counts else 0)

df = pd.DataFrame(scenario_tag_stats)
df

,tag,total_frames,max_consecutive,min_consecutive,avg_consecutive
0,traversing_intersection,57786,245,1,70.384896
1,low_magnitude_speed,8344,152,1,11.125333
2,near_pedestrian_on_crosswalk,74407,184,1,1.330597
3,on_stopline_traffic_light,21460,401,2,51.095238
4,near_barrier_on_driveable,438,71,1,1.659091
...,...,...,...,...,...
62,accelerating_at_crosswalk,55,9,1,2.391304
63,following_lane_without_lead,1190,292,1,21.250000
64,starting_straight_traffic_light_intersection_t...,340,1,1,1.000000
65,on_stopline_stop_sign,5161,327,2,67.907895


In [54]:
df.sort_values(by="avg_consecutive", ascending=False, inplace=True)
df

,tag,total_frames,max_consecutive,min_consecutive,avg_consecutive
15,on_pickup_dropoff,78646,401,1,333.245763
49,stationary_at_traffic_light_without_lead,16020,400,2,258.387097
31,stationary,188367,401,1,211.648315
44,traversing_pickup_dropoff,39330,401,1,191.853659
10,on_carpark,850,357,12,121.428571
...,...,...,...,...,...
21,changing_lane_to_right,7,1,1,1.000000
59,changing_lane,22,1,1,1.000000
25,starting_straight_stop_sign_intersection_trave...,266,1,1,1.000000
64,starting_straight_traffic_light_intersection_t...,340,1,1,1.000000


In [55]:
right_turns = df[df["tag"] == "starting_right_turn"]
right_turns

,tag,total_frames,max_consecutive,min_consecutive,avg_consecutive
6,starting_right_turn,604,20,4,11.843137


In [56]:
left_turns = df[df["tag"] == "starting_left_turn"]
left_turns


,tag,total_frames,max_consecutive,min_consecutive,avg_consecutive
26,starting_left_turn,600,20,2,11.111111


In [58]:
from collections import Counter

count_frequency = Counter(consecutive_counts["starting_right_turn"])
sorted_counts = sorted(count_frequency.items())
counts = [count for count, _ in sorted_counts]
frequencies = [freq for _, freq in sorted_counts]

fig = go.Figure()
fig.add_trace(go.Bar(x=counts, y=frequencies, name='Frequency of Consecutive Frames with Right Turn Scenario Tag'))
fig.update_layout(
    title='Consecutive Frames with Right Turn Scenario Tag', 
    xaxis_title='Consecutive Frames', 
    yaxis_title='Frequency'
)
fig.show()
